
# Embryo Diffusion — Phase-Guided & Presence-Filtered (Local Project)

This notebook trains a **conditional diffusion model** to predict future embryo frames.  
It integrates:
- **Embryo presence filtering**: only trains on images where an embryo is detected (via your presence classifier).
- **Phase guidance**: uses your **phase classifier** to condition the diffusion model on the current/target phase.

> Project structure assumptions (adapt if needed):
```
thesis/
├─ old notebooks/
│   ├─ embryo_diffusion_old.pdf
│   ├─ embryo_phase_classifier_old.pdf
│   └─ embryo_presence_classifier_old.pdf
├─ data/
│   ├─ embryo_dataset/               # F0 plane frames per embryo folder
│   │   ├─ AA83-7/
│   │   │   ├─ D2013...WELL7_RUN1.jpeg
│   │   │   └─ ...
│   │   └─ AAL839-6/...
│   └─ embryo_dataset_annotations/   # one CSV per embryo <ID>_phases.csv
│       ├─ AA83-7_phases.csv
│       └─ AAL839-6_phases.csv
├─ preprocessing/
│   ├─ train_presence_classifier.ipynb
│   ├─ train_phase_classifier.ipynb
│   ├─ no_embryo_models/             # your saved presence models
│   └─ embryo_classifier_models/     # your saved phase classifier models
└─ diffusion/
    ├─ train_diffusion.ipynb         # <- this notebook
    ├─ predict_next_frames.ipynb
    └─ diffusion_models/             # trained diffusion checkpoints
```
**Dataset**: 704 embryos, 16 annotated phases, F0 images 500×500, CSV labels with (phase, start_frame, end_frame). See the paper for details.  



> **Notes**
> - Presence model should output a probability that an image contains an embryo. We will keep frames with `p ≥ PRESENCE_THRESH`.
> - Phase model should output one of **16 phases** (`pPB2, pPNa, pPNf, p2, p3, p4, p5, p6, p7, p8, p9+, pM, pSB, pB, pEB, pHB`).
> - The diffusion UNet is **phase-conditioned** (classifier-free guidance optional).
> - Training data: pairs **(x_t, x_{t+Δ})** are sampled within each embryo video using the CSV ranges.


In [1]:

# If you need to install anything, do it here.
# !pip install torch torchvision pandas pillow tqdm numpy
# Optional (for SSIM/LPIPS):
# !pip install scikit-image lpips
import os, sys, glob, math, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set a deterministic seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


Using device: cuda


In [2]:

# ==== CONFIG ====
# Adjust these paths to your local project root
PROJECT_ROOT = Path.cwd().resolve()  # if you open this notebook in thesis/diffusion
DATA_ROOT = PROJECT_ROOT.parent / 'data'
IMG_ROOT = DATA_ROOT / 'embryo_dataset'                    # F0 plane images
ANN_ROOT = DATA_ROOT / 'embryo_dataset_annotations'        # *_phases.csv

PRESENCE_MODEL_DIR = PROJECT_ROOT.parent / 'preprocessing' / 'embryo_presence_models'
PHASE_MODEL_DIR    = PROJECT_ROOT.parent / 'preprocessing' / 'embryo_phase_models'

OUT_DIR   = PROJECT_ROOT / 'diffusion_models'
CACHE_DIR = PROJECT_ROOT / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Training
IMG_SIZE = 256           # we will center-crop/resize to this
BATCH_SIZE = 8
LR = 2e-4
EPOCHS = 10
NUM_WORKERS = 4
GRAD_ACCUM = 1
VAL_SPLIT = 0.05
MAX_SAMPLES_PER_EMBRYO = 150   # cap pairs per embryo to balance

# Pair sampling
DELTA_MIN = 1    # min frame gap
DELTA_MAX = 5    # max frame gap

# Presence filtering
PRESENCE_THRESH = 0.5
PRESENCE_MODEL_PATH = None  # set automatically from dir if found

# Phase classifier
PHASE_MODEL_PATH = None  # set automatically from dir if found
PHASE_LABELS = ['pPB2','pPNa','pPNf','p2','p3','p4','p5','p6','p7','p8','p9+','pM','pSB','pB','pEB','pHB']
NUM_PHASES = len(PHASE_LABELS)

# Diffusion
TIMESTEPS = 1000
BETA_START, BETA_END = 1e-4, 0.02
GUIDANCE_PROB = 0.9     # classifier-free guidance dropout (prob to KEEP conditioning)
GUIDANCE_SCALE = 2.0    # scale at sampling time

# Check existence
assert IMG_ROOT.exists(), f"Images folder not found: {IMG_ROOT}"
assert ANN_ROOT.exists(), f"Annotations folder not found: {ANN_ROOT}"
print('Project root:', PROJECT_ROOT)
print('Data root:', DATA_ROOT)


Project root: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/diffusion
Data root: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/data


In [3]:

def load_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')  # grayscale
    # Center-crop to square then resize to size x size
    w, h = img.size
    m = min(w, h)
    left = (w - m) // 2
    top = (h - m) // 2
    img = img.crop((left, top, left + m, top + m))
    if size is not None and (img.size[0] != size or img.size[1] != size):
        img = img.resize((size, size), Image.BICUBIC)
    arr = np.array(img, dtype=np.float32) / 255.0
    # Normalize to [-1, 1]
    arr = arr * 2 - 1
    # [1, H, W]
    return torch.from_numpy(arr).unsqueeze(0)

def denorm(x):
    # x in [-1, 1] -> [0, 1]
    return (x.clamp(-1,1) + 1) / 2


In [8]:

# We try to auto-detect a presence model file.
def autodetect_model(model_dir: Path, keywords=('presence','no_embryo','best','model','.pt','.pth','.ckpt')):
    if not model_dir.exists():
        return None
    cands = []
    for p in model_dir.glob('**/*'):
        name = p.name.lower()
        if p.is_file() and any(k in name for k in keywords):
            cands.append(p)
    if cands:
        # pick latest modified
        cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        return cands[0]
    return None

if PRESENCE_MODEL_PATH is None:
    PRESENCE_MODEL_PATH = autodetect_model(PRESENCE_MODEL_DIR)

print('Presence model path:', PRESENCE_MODEL_PATH)

class PresenceWrapper(nn.Module):
    """Wraps a user-provided binary classifier that returns probability of embryo presence.
    Expect it to accept [B,1,H,W] inputs in [-1,1] and return logits or prob.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        with torch.no_grad():
            out = self.model(x)
            if isinstance(out, (list, tuple)):
                out = out[0]
        # Map to probability via sigmoid if needed
        if out.shape[-1] == 1 or out.ndim==2:
            out = torch.sigmoid(out if out.ndim==2 else out.squeeze(-1))
        return out.squeeze()

def load_presence_model(path):
    if path is None:
        raise FileNotFoundError("Presence model not found. Please set PRESENCE_MODEL_PATH.")
    obj = torch.load(path, weights_only=True, map_location='cpu')
    # If a whole model was saved (recommended)
    if isinstance(obj, nn.Module):
        model = obj
    elif isinstance(obj, dict) and 'model' in obj and isinstance(obj['model'], nn.Module):
        model = obj['model']
    else:
        raise RuntimeError("Unsupported presence model format. Save full nn.Module in the checkpoint.")
    model.eval()
    return PresenceWrapper(model)

try:
    presence_model = load_presence_model(PRESENCE_MODEL_PATH).to(device) if PRESENCE_MODEL_PATH else None
    print('Presence model loaded.' if presence_model else 'Presence model not loaded.')
except Exception as e:
    print('Presence model load error:', e)
    presence_model = None


Presence model path: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/preprocessing/embryo_presence_models/embryo_presence_resnet18.pth
Presence model load error: Unsupported presence model format. Save full nn.Module in the checkpoint.


In [10]:

if PHASE_MODEL_PATH is None:
    PHASE_MODEL_PATH = autodetect_model(PHASE_MODEL_DIR, keywords=('phase','classifier','best','model','.pt','.pth','.ckpt'))

print('Phase model path:', PHASE_MODEL_PATH)

class PhaseWrapper(nn.Module):
    """Wraps a user-provided multi-class phase classifier (16 classes)."""
    def __init__(self, model, num_classes=16):
        super().__init__()
        self.model = model
        self.num_classes = num_classes

    def forward(self, x):
        with torch.no_grad():
            logits = self.model(x)
            if isinstance(logits, (list,tuple)):
                logits = logits[0]
        return logits

def load_phase_model(path):
    if path is None:
        raise FileNotFoundError("Phase model not found. Please set PHASE_MODEL_PATH.")
    obj = torch.load(path, weights_only=True, map_location='cpu')
    if isinstance(obj, nn.Module):
        model = obj
    elif isinstance(obj, dict) and 'model' in obj and isinstance(obj['model'], nn.Module):
        model = obj['model']
    else:
        raise RuntimeError("Unsupported phase model format. Save full nn.Module in the checkpoint.")
    model.eval()
    return PhaseWrapper(model, num_classes=NUM_PHASES)

try:
    phase_model = load_phase_model(PHASE_MODEL_PATH).to(device) if PHASE_MODEL_PATH else None
    print('Phase model loaded.' if phase_model else 'Phase model not loaded.')
except Exception as e:
    print('Phase model load error:', e)
    phase_model = None


Phase model path: /mnt/c/Users/ioann/OneDrive/Documents/Projects/thesis/preprocessing/embryo_phase_models/phase_classifier_20251009_225010.pth
Phase model load error: Unsupported phase model format. Save full nn.Module in the checkpoint.


In [11]:

INDEX_CACHE = CACHE_DIR / 'frame_index.csv'

def build_index():
    rows = []
    embryo_ids = sorted([d.name for d in IMG_ROOT.iterdir() if d.is_dir()])
    for eid in tqdm(embryo_ids, desc='Indexing embryos'):
        ann_path = ANN_ROOT / f"{eid}_phases.csv"
        if not ann_path.exists():
            continue
        img_files = sorted(glob.glob(str(IMG_ROOT / eid / '*.jpeg')))
        if not img_files:
            img_files = sorted(glob.glob(str(IMG_ROOT / eid / '*.jpg')))
        # Build a vector frame->phase_id using CSV
        df = pd.read_csv(ann_path, header=None, names=['phase','start','end'], sep='[,;\s]+', engine='python')
        # Map phase string to class id
        def phase_to_id(ph):
            if ph in PHASE_LABELS:
                return PHASE_LABELS.index(ph)
            # handle tPB2 vs pPB2 etc (paper uses t* as events; frames labelled p*). Accept both.
            ph = ph.strip()
            ph = ph.replace('t','p') if ph.startswith('t') else ph
            return PHASE_LABELS.index(ph)
        label_spans = []
        for _,r in df.iterrows():
            try:
                pid = phase_to_id(str(r['phase']))
                label_spans.append((int(r['start']), int(r['end']), pid))
            except Exception:
                continue
        # Assign phase to frames
        for i, f in enumerate(img_files):
            # Find the most recent span that includes i
            pid = None
            for s,e,p in label_spans:
                if s <= i <= e:
                    pid = p
                    break
            if pid is None:
                # Before first event -> skip
                continue
            rows.append({'embryo_id': eid, 'frame_idx': i, 'path': f, 'phase_id': pid})
    return pd.DataFrame(rows)

if INDEX_CACHE.exists():
    index_df = pd.read_csv(INDEX_CACHE)
else:
    index_df = build_index()
    index_df.to_csv(INDEX_CACHE, index=False)

print('Indexed frames:', len(index_df))
index_df.head()


Indexed frames: 297138


,embryo_id,frame_idx,path,phase_id
0,AA83-7,5,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
1,AA83-7,6,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
2,AA83-7,7,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
3,AA83-7,8,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0
4,AA83-7,9,/mnt/c/Users/ioann/OneDrive/Documents/Projects...,0


In [12]:

# Apply presence filtering (cache)
PRESENCE_CACHE = CACHE_DIR / f'presence_mask_thr{PRESENCE_THRESH:.2f}.csv'

def compute_presence_mask(df: pd.DataFrame):
    assert presence_model is not None, "Presence model required for filtering. Please load it."
    paths = df['path'].tolist()
    mask = []
    bs = 64
    with torch.no_grad():
        for i in tqdm(range(0, len(paths), bs), desc='Running presence model'):
            batch_paths = paths[i:i+bs]
            imgs = torch.stack([load_image(p) for p in batch_paths]).to(device)
            probs = presence_model(imgs).detach().float().cpu().numpy().tolist()
            mask.extend([int(p>=PRESENCE_THRESH) for p in probs])
    out = df.copy()
    out['is_present'] = mask
    return out

if PRESENCE_CACHE.exists():
    present_df = pd.read_csv(PRESENCE_CACHE)
else:
    if presence_model is None:
        raise RuntimeError("Presence model missing. Set PRESENCE_MODEL_PATH or place a model in preprocessing/no_embryo_models.")
    present_df = compute_presence_mask(index_df)
    present_df.to_csv(PRESENCE_CACHE, index=False)

print('Frames after presence inference:', present_df['is_present'].sum(), '/', len(present_df))
present_df.head()


RuntimeError: Presence model missing. Set PRESENCE_MODEL_PATH or place a model in preprocessing/no_embryo_models.

In [ ]:

def build_pairs(df: pd.DataFrame, delta_min=DELTA_MIN, delta_max=DELTA_MAX, max_samples_per_embryo=MAX_SAMPLES_PER_EMBRYO):
    pairs = []
    kept = df[df['is_present']==1]
    by_embryo = kept.groupby('embryo_id')
    for eid, g in tqdm(by_embryo, desc='Sampling pairs per embryo'):
        g = g.sort_values('frame_idx')
        files = g['path'].tolist()
        phases = g['phase_id'].tolist()
        n = len(files)
        if n < delta_min+1:
            continue
        # sample pairs
        cnt = 0
        tries = 0
        while cnt < max_samples_per_embryo and tries < max(1000, 5*max_samples_per_embryo):
            tries += 1
            i = random.randrange(0, n-1)
            d = random.randint(delta_min, min(delta_max, n-1-i))
            j = i + d
            # require both present
            pi, pj = phases[i], phases[j]
            pairs.append({'embryo_id': eid, 'src_path': files[i], 'tgt_path': files[j], 'src_phase': pi, 'tgt_phase': pj})
            cnt += 1
    return pd.DataFrame(pairs)

PAIRS_CACHE = CACHE_DIR / f'pairs_d{DELTA_MIN}-{DELTA_MAX}_max{MAX_SAMPLES_PER_EMBRYO}.csv'
if PAIRS_CACHE.exists():
    pairs_df = pd.read_csv(PAIRS_CACHE)
else:
    pairs_df = build_pairs(present_df)
    pairs_df.to_csv(PAIRS_CACHE, index=False)

print('Total training pairs:', len(pairs_df))
pairs_df.head()


In [ ]:

# Simple split by embryo_id to prevent leakage
embryos = pairs_df['embryo_id'].unique().tolist()
random.shuffle(embryos)
val_count = max(1, int(len(embryos)*VAL_SPLIT))
val_embryos = set(embryos[:val_count])
train_df = pairs_df[~pairs_df['embryo_id'].isin(val_embryos)].reset_index(drop=True)
val_df   = pairs_df[pairs_df['embryo_id'].isin(val_embryos)].reset_index(drop=True)
print(f'Embryos total={len(embryos)}, train={len(embryos)-val_count}, val={val_count}')
print('Train pairs:', len(train_df), 'Val pairs:', len(val_df))


In [ ]:

class PairDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x_src = load_image(row['src_path'])
        x_tgt = load_image(row['tgt_path'])
        src_phase = int(row['src_phase'])
        tgt_phase = int(row['tgt_phase'])
        return x_src, x_tgt, src_phase, tgt_phase

train_loader = DataLoader(PairDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(PairDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)


In [ ]:

def make_beta_schedule(timesteps=TIMESTEPS, beta_start=BETA_START, beta_end=BETA_END):
    return torch.linspace(beta_start, beta_end, timesteps)

betas = make_beta_schedule()
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), alphas_cumprod[:-1]], dim=0)

to_dev = lambda t: t.to(device).float()
betas = to_dev(betas); alphas = to_dev(alphas); alphas_cumprod = to_dev(alphas_cumprod); alphas_cumprod_prev = to_dev(alphas_cumprod_prev)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)


In [ ]:

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_emb_ch, cond_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time = nn.Linear(t_emb_ch, out_ch)
        self.cond = nn.Linear(cond_ch, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb, c_emb):
        h = self.conv1(F.silu(x))
        h = h + self.time(F.silu(t_emb))[:,:,None,None] + self.cond(F.silu(c_emb))[:,:,None,None]
        h = self.conv2(F.silu(h))
        return h + self.skip(x)

def timestep_embedding(timesteps, dim=128):
    # sinusoidal
    half = dim//2
    freqs = torch.exp(-math.log(10000) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / (half-1))
    args = timesteps.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
    return emb

class UNetCond(nn.Module):
    def __init__(self, in_ch=1, base=64, t_emb_dim=128, cond_dim=128, num_classes=NUM_PHASES):
        super().__init__()
        self.class_emb = nn.Embedding(num_classes+1, cond_dim)  # +1 for CFG null token
        self.null_class = num_classes

        self.t_mlp = nn.Sequential(nn.Linear(t_emb_dim, t_emb_dim), nn.SiLU(), nn.Linear(t_emb_dim, t_emb_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))

        self.inp = nn.Conv2d(in_ch, base, 3, padding=1)
        self.rb1 = ResBlock(base, base, t_emb_dim, cond_dim)
        self.down1 = nn.Conv2d(base, base*2, 4, stride=2, padding=1)
        self.rb2 = ResBlock(base*2, base*2, t_emb_dim, cond_dim)
        self.down2 = nn.Conv2d(base*2, base*4, 4, stride=2, padding=1)
        self.rb3 = ResBlock(base*4, base*4, t_emb_dim, cond_dim)
        self.mid = ResBlock(base*4, base*4, t_emb_dim, cond_dim)
        self.up1 = nn.ConvTranspose2d(base*4, base*2, 4, stride=2, padding=1)
        self.rb4 = ResBlock(base*2, base*2, t_emb_dim, cond_dim)
        self.up2 = nn.ConvTranspose2d(base*2, base, 4, stride=2, padding=1)
        self.rb5 = ResBlock(base, base, t_emb_dim, cond_dim)
        self.out = nn.Conv2d(base, in_ch, 3, padding=1)

    def forward(self, x, t, cls):
        # cls: [B] class ids, may include null for CFG
        t_emb = timestep_embedding(t, dim=128)
        t_emb = self.t_mlp(t_emb)
        c_emb = self.class_emb(cls.clamp(0, self.class_emb.num_embeddings-1))
        c_emb = self.c_mlp(c_emb)

        h = self.inp(x)
        h1 = self.rb1(h, t_emb, c_emb)
        h2 = self.rb2(self.down1(h1), t_emb, c_emb)
        h3 = self.rb3(self.down2(h2), t_emb, c_emb)
        hmid = self.mid(h3, t_emb, c_emb)
        u1 = self.rb4(self.up1(hmid), t_emb, c_emb) + h2
        u2 = self.rb5(self.up2(u1), t_emb, c_emb) + h1
        out = self.out(F.silu(u2))
        return out


In [ ]:

class DiffusionCond(nn.Module):
    def __init__(self, model: UNetCond, timesteps=TIMESTEPS):
        super().__init__()
        self.model = model
        self.timesteps = timesteps

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha = sqrt_alphas_cumprod[t][:,None,None,None]
        sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t][:,None,None,None]
        return sqrt_alpha * x0 + sqrt_one_minus * noise, noise

    def p_losses(self, x0, cond_cls):
        B = x0.size(0)
        t = torch.randint(0, self.timesteps, (B,), device=x0.device, dtype=torch.long)
        # add noise
        x_noisy, noise = self.q_sample(x0, t)
        # classifier-free guidance dropout on conditioning
        use_cond = (torch.rand(B, device=x0.device) < GUIDANCE_PROB).long()
        cls_in = cond_cls.clone()
        cls_in[use_cond==0] = self.model.null_class
        pred = self.model(x_noisy, t, cls_in)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def p_sample(self, x, t, cls, guidance_scale=GUIDANCE_SCALE):
        # two passes: cond and null for CFG
        null = torch.full_like(cls, self.model.null_class)
        eps_cond = self.model(x, t, cls)
        eps_null = self.model(x, t, null)
        eps = eps_null + guidance_scale * (eps_cond - eps_null)

        beta_t = betas[t][:,None,None,None]
        sqrt_one_minus = sqrt_one_minus_alphas_cumprod[t][:,None,None,None]
        sqrt_recip = sqrt_recip_alphas[t][:,None,None,None]
        mean = sqrt_recip * (x - beta_t / sqrt_one_minus * eps)
        if t.min() > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)
        var = posterior_variance[t][:,None,None,None]
        return mean + torch.sqrt(var) * noise

    @torch.no_grad()
    def sample(self, shape, cls, x_start=None):
        B = shape[0]
        if x_start is None:
            x = torch.randn(shape, device=device)
        else:
            # start from noise; we use x_start as an *additional input conditioning* by concatenating channels
            # For simplicity here, we integrate x_start via replacing initial x with Gaussian (standard sampling).
            x = torch.randn(shape, device=device)
        for i in reversed(range(self.timesteps)):
            t = torch.full((B,), i, device=device, dtype=torch.long)
            x = self.p_sample(x, t, cls)
        return x


In [ ]:

model = UNetCond(in_ch=1, base=64, num_classes=NUM_PHASES).to(device)
diffusion = DiffusionCond(model, timesteps=TIMESTEPS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

global_step = 0
best_val = None
ckpt_path = OUT_DIR / f'phase_guided_ddpm_{IMG_SIZE}px.pt'

def one_epoch(loader, train=True):
    global global_step, best_val
    model.train(mode=train)
    total = 0.0
    count = 0
    for x_src, x_tgt, src_phase, tgt_phase in tqdm(loader, disable=False):
        x_src = x_src.to(device, non_blocking=True)   # not directly used in this minimal UNet; placeholder for future concat conditioning
        x_tgt = x_tgt.to(device, non_blocking=True)
        # We condition on the *target* phase here (you can switch to source or both with concatenation ideas)
        cond_cls = torch.tensor(tgt_phase, device=device, dtype=torch.long)

        loss = diffusion.p_losses(x_tgt, cond_cls)

        if train:
            loss.backward()
            if (global_step+1) % GRAD_ACCUM == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            global_step += 1

        total += loss.item() * x_tgt.size(0)
        count += x_tgt.size(0)

    return total / max(1,count)

for epoch in range(1, EPOCHS+1):
    tl = one_epoch(train_loader, train=True)
    vl = one_epoch(val_loader, train=False)
    print(f"[Epoch {epoch:03d}] train {tl:.4f} | val {vl:.4f}")
    # Save best
    if best_val is None or vl < best_val:
        best_val = vl
        torch.save({'model': model, 'epoch': epoch, 'val_loss': vl, 'config': {
            'IMG_SIZE': IMG_SIZE, 'TIMESTEPS': TIMESTEPS, 'PHASE_LABELS': PHASE_LABELS
        }}, ckpt_path)
        print('Saved best to', ckpt_path)


In [ ]:

@torch.no_grad()
def visualize_samples(num=4):
    samp = val_df.sample(n=min(num, len(val_df))).reset_index(drop=True)
    imgs = []
    for i in range(len(samp)):
        row = samp.iloc[i]
        x_src = load_image(row['src_path']).to(device)[None]
        tgt_cls = torch.tensor([int(row['tgt_phase'])], device=device, dtype=torch.long)
        x_gen = diffusion.sample(shape=(1,1,IMG_SIZE,IMG_SIZE), cls=tgt_cls)
        imgs.append((x_src[0].cpu(), x_gen[0].cpu()))
    # show grid with PIL
    import matplotlib.pyplot as plt
    for i,(src,gen) in enumerate(imgs):
        plt.figure()
        plt.subplot(1,2,1); plt.title('Source (x_t)'); plt.imshow(denorm(src.squeeze()).cpu(), cmap='gray'); plt.axis('off')
        plt.subplot(1,2,2); plt.title('Predicted (x_{t+Δ})'); plt.imshow(denorm(gen.squeeze()).cpu(), cmap='gray'); plt.axis('off')
        plt.show()

# visualize_samples(4)


In [ ]:

def save_tensor_image(tensor, path):
    arr = (denorm(tensor).clamp(0,1).cpu().numpy()*255).astype(np.uint8)
    if arr.ndim==3 and arr.shape[0]==1:
        arr = arr[0]
    img = Image.fromarray(arr, mode='L')
    img.save(path)

@torch.no_grad()
def predict_next_frames(input_image_path, target_phase: str, out_path):
    assert target_phase in PHASE_LABELS, f"Unknown phase {target_phase}. Choices: {PHASE_LABELS}"
    cls = torch.tensor([PHASE_LABELS.index(target_phase)], device=device, dtype=torch.long)
    x_gen = diffusion.sample(shape=(1,1,IMG_SIZE,IMG_SIZE), cls=cls)
    save_tensor_image(x_gen[0], out_path)
    return out_path

# Example:
# predict_next_frames('/path/to/frame.jpeg', 'pB', 'predicted.png')


_Notebook generated on 2025-10-19 14:54:54._